In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score
)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.14.0+cu130
CUDA available: True
Using device: cuda
GPU: NVIDIA GeForce RTX 2050


In [2]:
train = pd.read_csv("../data/processed/train.csv")
test = pd.read_csv("../data/processed/test.csv")

print("Train shape:", train.shape)
print("Test shape:", test.shape)

Train shape: (5090096, 20)
Test shape: (1272524, 20)


In [3]:
print(train["isFraud"].value_counts())
print(test["isFraud"].value_counts())

isFraud
0    5083526
1       6570
Name: count, dtype: int64
isFraud
0    1270881
1       1643
Name: count, dtype: int64


In [4]:
X_train = train.drop(columns=["isFraud"])
y_train = train["isFraud"]

X_test = test.drop(columns=["isFraud"])
y_test = test["isFraud"]

X_train_d1 = X_train.sample(
    n=1_000_000,
    random_state =42
)
y_train_d1 = y_train.loc[X_train_d1.index]

print("DL training shape:", X_train_d1.shape)
print("DL target shape:", y_train_d1.shape)

print("\nClass distribution:")
print(y_train_d1.value_counts())


DL training shape: (1000000, 19)
DL target shape: (1000000,)

Class distribution:
isFraud
0    998783
1      1217
Name: count, dtype: int64


In [5]:
X_train_np = X_train_d1.values.astype(np.float32)
y_train_np = y_train_d1.values.astype(np.float32)

X_test_np = X_test.values.astype(np.float32)
y_test_np = y_test.values.astype(np.float32)

X_train_tensor =torch.from_numpy(X_train_np)
y_train_tensor = torch.from_numpy(y_train_np)

X_test_tensor = torch.from_numpy(X_test_np)
y_test_tensor = torch.from_numpy(y_test_np)

print("X_train tensor:", X_train_tensor.shape)
print("y_train tensor:", y_train_tensor.shape)
print("X_test tensor:", X_test_tensor.shape)
print("y_test tensor:", y_test_tensor.shape)


X_train tensor: torch.Size([1000000, 19])
y_train tensor: torch.Size([1000000])
X_test tensor: torch.Size([1272524, 19])
y_test tensor: torch.Size([1272524])


In [6]:
X_train_tensor , X_val_tensor, y_train_tensor,y_val_tensor = train_test_split(
    X_train_tensor,
    y_train_tensor,
    random_state=42,
    test_size=0.10,
    stratify=y_train_tensor
)
print("Training:", X_train_tensor.shape)
print("Validation:", X_val_tensor.shape)

print("\nTraining fraud cases:", int(y_train_tensor.sum()))
print("Validation fraud cases:", int(y_val_tensor.sum()))

Training: torch.Size([900000, 19])
Validation: torch.Size([100000, 19])

Training fraud cases: 1095
Validation fraud cases: 122


In [7]:
from torch.utils.data import TensorDataset, DataLoader

# Create datasets
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)

# Create DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=2048,
    shuffle=True,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=2048,
    shuffle=False,
    pin_memory=True
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))

Training batches: 440
Validation batches: 49


In [8]:
class FraudDetectionNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(19,64),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(64,32),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(32,1)
        )
    def forward(self,x):
        return self.network(x)
    

model = FraudDetectionNN().to(device)
print(model)

FraudDetectionNN(
  (network): Sequential(
    (0): Linear(in_features=19, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=64, out_features=32, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=32, out_features=1, bias=True)
  )
)


In [9]:
num_legitimate = (y_train_tensor == 0).sum().item()
num_fraud = (y_train_tensor == 1).sum().item()

pos_weight = num_legitimate / num_fraud
print(num_legitimate)
print(num_fraud)
print(pos_weight)

898905
1095
820.917808219178


In [10]:
pos_weight_tensor = torch.tensor(
    [pos_weight],
    dtype=torch.float32,
    device = device
)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)
optimizer = optim.Adam(model.parameters(), lr=0.001)
print("Loss function:", criterion)
print("Optimizer:", optimizer)

Loss function: BCEWithLogitsLoss()
Optimizer: Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)


In [11]:
num_epochs = 10

for epoch in range(num_epochs):

    # -------------------------
    # Training
    # -------------------------
    model.train()

    train_loss = 0.0

    for X_batch, y_batch in train_loader:

        # Move batch to GPU
        X_batch = X_batch.to(device, non_blocking=True)
        y_batch = y_batch.to(device, non_blocking=True).unsqueeze(1)

        # Clear previous gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(X_batch)

        # Calculate loss
        loss = criterion(outputs, y_batch)

        # Backpropagation
        loss.backward()

        # Update weights
        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    # -------------------------
    # Validation
    # -------------------------
    model.eval()

    val_loss = 0.0

    with torch.no_grad():

        for X_batch, y_batch in val_loader:

            X_batch = X_batch.to(device, non_blocking=True)
            y_batch = y_batch.to(device, non_blocking=True).unsqueeze(1)

            outputs = model(X_batch)

            loss = criterion(outputs, y_batch)

            val_loss += loss.item()

    val_loss /= len(val_loader)

    print(
        f"Epoch [{epoch + 1}/{num_epochs}] "
        f"Train Loss: {train_loss:.4f} "
        f"Val Loss: {val_loss:.4f}"
    )

Epoch [1/10] Train Loss: 13107.1517 Val Loss: 2044.5715
Epoch [2/10] Train Loss: 3888.0745 Val Loss: 1345.0821
Epoch [3/10] Train Loss: 2187.5863 Val Loss: 1467.5004
Epoch [4/10] Train Loss: 2510.0437 Val Loss: 803.9201
Epoch [5/10] Train Loss: 439.1928 Val Loss: 469.3227
Epoch [6/10] Train Loss: 402.8650 Val Loss: 255.1809
Epoch [7/10] Train Loss: 300.9862 Val Loss: 168.2431
Epoch [8/10] Train Loss: 79.9195 Val Loss: 113.1115
Epoch [9/10] Train Loss: 54.5606 Val Loss: 79.8656
Epoch [10/10] Train Loss: 12.2842 Val Loss: 80.9829


In [12]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score
)

model.eval()

all_probabilities = []

with torch.no_grad():

    # Process test data in batches
    test_loader = DataLoader(
        TensorDataset(X_test_tensor, y_test_tensor),
        batch_size=2048,
        shuffle=False,
        pin_memory=True
    )

    for X_batch, _ in test_loader:

        X_batch = X_batch.to(device, non_blocking=True)

        outputs = model(X_batch)

        # Convert logits → probabilities
        probabilities = torch.sigmoid(outputs)

        all_probabilities.append(
            probabilities.cpu().numpy()
        )

# Combine all batches
y_test_prob = np.concatenate(all_probabilities).ravel()

# Default threshold
y_test_pred = (y_test_prob >= 0.5).astype(int)

print("ROC-AUC:", roc_auc_score(y_test, y_test_prob))
print("PR-AUC:", average_precision_score(y_test, y_test_prob))

print("\nClassification Report:")
print(classification_report(y_test, y_test_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))

ROC-AUC: 0.8872438822126048
PR-AUC: 0.36695717443211967

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00   1270881
           1       0.14      0.48      0.22      1643

    accuracy                           1.00   1272524
   macro avg       0.57      0.74      0.61   1272524
weighted avg       1.00      1.00      1.00   1272524


Confusion Matrix:
[[1265982    4899]
 [    854     789]]


In [13]:
from sklearn.preprocessing import StandardScaler
X_train_d1_np = X_train_tensor.cpu().numpy()
scaler_d1 = StandardScaler()

X_train_scaled = scaler_d1.fit_transform(X_train_d1_np)
X_val_scaled = scaler_d1.transform(X_val_tensor.cpu().numpy())

X_test_scaled = scaler_d1.transform(X_test_tensor.cpu().numpy())

print("Training:", X_train_scaled.shape)
print("Validation:", X_val_scaled.shape)
print("Test:", X_test_scaled.shape)


Training: (900000, 19)
Validation: (100000, 19)
Test: (1272524, 19)


In [14]:
X_train_tensor = torch.tensor(X_train_scaled,dtype=torch.float32)
X_val_tensor = torch.tensor(X_val_scaled,dtype = torch.float32)
X_test_tensor = torch.tensor(X_test_scaled,dtype = torch.float32)


print(X_train_tensor.shape)
print(X_val_tensor.shape)
print(X_test_tensor.shape)

torch.Size([900000, 19])
torch.Size([100000, 19])
torch.Size([1272524, 19])


In [15]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor
)

val_dataset = TensorDataset(
    X_val_tensor,
    y_val_tensor
)

train_loader = DataLoader(
    train_dataset,
    batch_size=2048,
    shuffle=True,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=2048,
    shuffle=False,
    pin_memory=True
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))

Training batches: 440
Validation batches: 49


In [16]:
model = FraudDetectionNN().to(device)

print(model)
num_legitimate = (y_train_tensor == 0).sum().item()
num_fraud = (y_train_tensor == 1).sum().item()

pos_weight = num_legitimate / num_fraud

pos_weight_tensor = torch.tensor(
    [pos_weight],
    dtype=torch.float32,
    device=device
)

criterion = nn.BCEWithLogitsLoss(
    pos_weight=pos_weight_tensor
)

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

print("pos_weight:", pos_weight)

FraudDetectionNN(
  (network): Sequential(
    (0): Linear(in_features=19, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=64, out_features=32, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=32, out_features=1, bias=True)
  )
)
pos_weight: 820.917808219178


In [17]:
num_epochs = 10

for epoch in range(num_epochs):

    model.train()
    train_loss = 0.0

    for X_batch, y_batch in train_loader:

        X_batch = X_batch.to(device, non_blocking=True)
        y_batch = y_batch.to(device, non_blocking=True).unsqueeze(1)

        optimizer.zero_grad()

        outputs = model(X_batch)

        loss = criterion(outputs, y_batch)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    # Validation
    model.eval()
    val_loss = 0.0

    with torch.no_grad():

        for X_batch, y_batch in val_loader:

            X_batch = X_batch.to(device, non_blocking=True)
            y_batch = y_batch.to(device, non_blocking=True).unsqueeze(1)

            outputs = model(X_batch)

            loss = criterion(outputs, y_batch)

            val_loss += loss.item()

    val_loss /= len(val_loader)

    print(
        f"Epoch [{epoch + 1}/{num_epochs}] "
        f"Train Loss: {train_loss:.4f} "
        f"Val Loss: {val_loss:.4f}"
    )

Epoch [1/10] Train Loss: 0.4738 Val Loss: 0.3136
Epoch [2/10] Train Loss: 0.2469 Val Loss: 0.3117
Epoch [3/10] Train Loss: 0.2010 Val Loss: 0.3121
Epoch [4/10] Train Loss: 0.2015 Val Loss: 0.3284
Epoch [5/10] Train Loss: 0.1944 Val Loss: 0.3295
Epoch [6/10] Train Loss: 0.1832 Val Loss: 0.3322
Epoch [7/10] Train Loss: 0.1764 Val Loss: 0.3245
Epoch [8/10] Train Loss: 0.1757 Val Loss: 0.3329
Epoch [9/10] Train Loss: 0.1632 Val Loss: 0.3428
Epoch [10/10] Train Loss: 0.1618 Val Loss: 0.3393


In [18]:
model.eval()

test_loader = DataLoader(
    TensorDataset(X_test_tensor, y_test_tensor),
    batch_size=2048,
    shuffle=False,
    pin_memory=True
)

all_probabilities = []

with torch.no_grad():

    for X_batch, _ in test_loader:

        X_batch = X_batch.to(device, non_blocking=True)

        outputs = model(X_batch)

        probabilities = torch.sigmoid(outputs)

        all_probabilities.append(
            probabilities.cpu().numpy()
        )

y_test_prob = np.concatenate(all_probabilities).ravel()

# Default threshold
y_test_pred = (y_test_prob >= 0.5).astype(int)

print("ROC-AUC:", roc_auc_score(y_test, y_test_prob))
print("PR-AUC:", average_precision_score(y_test, y_test_prob))

print("\nClassification Report:")
print(classification_report(y_test, y_test_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))

ROC-AUC: 0.9964673575511905
PR-AUC: 0.7500232602406286

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.96      0.98   1270881
           1       0.03      0.99      0.06      1643

    accuracy                           0.96   1272524
   macro avg       0.51      0.97      0.52   1272524
weighted avg       1.00      0.96      0.98   1272524


Confusion Matrix:
[[1216894   53987]
 [     15    1628]]


In [19]:
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = [
    0.01, 0.05, 0.10, 0.15, 0.20,
    0.25, 0.30, 0.35, 0.40, 0.45,
    0.50, 0.55, 0.60, 0.65, 0.70,
    0.75, 0.80, 0.85, 0.90, 0.95
]

results = []

for threshold in thresholds:

    predictions = (y_test_prob >= threshold).astype(int)

    precision = precision_score(
        y_test,
        predictions,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        predictions,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        predictions,
        zero_division=0
    )

    results.append({
        "Threshold": threshold,
        "Precision": precision,
        "Recall": recall,
        "F1": f1
    })

threshold_results = pd.DataFrame(results)

print(threshold_results)


    Threshold  Precision    Recall        F1
0        0.01   0.008132  0.998174  0.016132
1        0.05   0.012018  0.997565  0.023749
2        0.10   0.014719  0.997565  0.029011
3        0.15   0.016480  0.996957  0.032424
4        0.20   0.017829  0.996348  0.035031
5        0.25   0.019057  0.995740  0.037399
6        0.30   0.020236  0.995131  0.039665
7        0.35   0.021717  0.995131  0.042507
8        0.40   0.023503  0.994522  0.045921
9        0.45   0.025854  0.992696  0.050395
10       0.50   0.029273  0.990870  0.056865
11       0.55   0.033785  0.982958  0.065325
12       0.60   0.038988  0.977480  0.074985
13       0.65   0.045303  0.964699  0.086541
14       0.70   0.053112  0.953135  0.100617
15       0.75   0.062696  0.936093  0.117521
16       0.80   0.077660  0.926963  0.143314
17       0.85   0.098744  0.914181  0.178237
18       0.90   0.133540  0.889836  0.232229
19       0.95   0.220259  0.849665  0.349831


In [20]:
best_row = threshold_results.loc[
    threshold_results["F1"].idxmax()
]

print("\nBest threshold:")
print(best_row)


Best threshold:
Threshold    0.950000
Precision    0.220259
Recall       0.849665
F1           0.349831
Name: 19, dtype: float64


In [21]:
model = FraudDetectionNN().to(device)

num_legitimate = (y_train_tensor == 0).sum().item()
num_fraud = (y_train_tensor == 1).sum().item()

pos_weight = num_legitimate / num_fraud

pos_weight_tensor = torch.tensor(
    [pos_weight],
    dtype=torch.float32,
    device=device
)

criterion = nn.BCEWithLogitsLoss(
    pos_weight=pos_weight_tensor
)

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

print("pos_weight:", pos_weight)
print("Device:", device)

pos_weight: 820.917808219178
Device: cuda


In [22]:
import copy

num_epochs = 20
patience = 3

best_val_loss = float("inf")
best_model_state = None
epochs_without_improvement = 0

for epoch in range(num_epochs):

    # =========================
    # TRAINING
    # =========================
    model.train()

    train_loss = 0.0

    for X_batch, y_batch in train_loader:

        X_batch = X_batch.to(device, non_blocking=True)
        y_batch = y_batch.to(device, non_blocking=True).unsqueeze(1)

        optimizer.zero_grad()

        outputs = model(X_batch)

        loss = criterion(outputs, y_batch)

        loss.backward()

        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    # =========================
    # VALIDATION
    # =========================
    model.eval()

    val_loss = 0.0

    with torch.no_grad():

        for X_batch, y_batch in val_loader:

            X_batch = X_batch.to(device, non_blocking=True)
            y_batch = y_batch.to(device, non_blocking=True).unsqueeze(1)

            outputs = model(X_batch)

            loss = criterion(outputs, y_batch)

            val_loss += loss.item()

    val_loss /= len(val_loader)

    print(
        f"Epoch [{epoch + 1}/{num_epochs}] "
        f"Train Loss: {train_loss:.4f} "
        f"Val Loss: {val_loss:.4f}"
    )

    # =========================
    # EARLY STOPPING
    # =========================

    if val_loss < best_val_loss:

        best_val_loss = val_loss

        best_model_state = copy.deepcopy(model.state_dict())

        epochs_without_improvement = 0

        print("  ✓ Best model saved")

    else:

        epochs_without_improvement += 1

        print(
            f"  No improvement "
            f"({epochs_without_improvement}/{patience})"
        )

        if epochs_without_improvement >= patience:

            print("\nEarly stopping triggered.")

            break


# Restore best model
model.load_state_dict(best_model_state)

print("\nBest validation loss:", best_val_loss)
print("Best model restored.")

Epoch [1/20] Train Loss: 0.5564 Val Loss: 0.3065
  ✓ Best model saved
Epoch [2/20] Train Loss: 0.2607 Val Loss: 0.2994
  ✓ Best model saved
Epoch [3/20] Train Loss: 0.2082 Val Loss: 0.3194
  No improvement (1/3)
Epoch [4/20] Train Loss: 0.2032 Val Loss: 0.3265
  No improvement (2/3)
Epoch [5/20] Train Loss: 0.1866 Val Loss: 0.3425
  No improvement (3/3)

Early stopping triggered.

Best validation loss: 0.2993715555996311
Best model restored.


In [23]:
model.eval()

test_loader = DataLoader(
    TensorDataset(X_test_tensor, y_test_tensor),
    batch_size=2048,
    shuffle=False,
    pin_memory=True
)

all_probabilities = []

with torch.no_grad():

    for X_batch, _ in test_loader:

        X_batch = X_batch.to(device, non_blocking=True)

        outputs = model(X_batch)

        probabilities = torch.sigmoid(outputs)

        all_probabilities.append(
            probabilities.cpu().numpy()
        )

# Combine predictions
y_test_prob = np.concatenate(all_probabilities).ravel()

# Default threshold
y_test_pred = (y_test_prob >= 0.5).astype(int)

print("ROC-AUC:", roc_auc_score(y_test, y_test_prob))
print("PR-AUC:", average_precision_score(y_test, y_test_prob))

print("\nClassification Report:")
print(classification_report(y_test, y_test_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))


ROC-AUC: 0.9936144789554148
PR-AUC: 0.6350889109710885

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.94      0.97   1270881
           1       0.02      0.98      0.04      1643

    accuracy                           0.94   1272524
   macro avg       0.51      0.96      0.50   1272524
weighted avg       1.00      0.94      0.97   1272524


Confusion Matrix:
[[1188496   82385]
 [     31    1612]]


In [24]:
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = np.arange(0.50, 1.00, 0.05)

results = []

for threshold in thresholds:
    y_pred = (y_test_prob >= threshold).astype(int)

    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    results.append({
        "Threshold": round(threshold, 2),
        "Precision": precision,
        "Recall": recall,
        "F1": f1
    })

threshold_results = pd.DataFrame(results)

print(threshold_results)
best_row = threshold_results.loc[
    threshold_results["F1"].idxmax()
]

print("\nBest DL Threshold:")
print(best_row)

   Threshold  Precision    Recall        F1
0       0.50   0.019191  0.981132  0.037646
1       0.55   0.022542  0.969568  0.044059
2       0.60   0.027621  0.959221  0.053696
3       0.65   0.034326  0.948874  0.066256
4       0.70   0.042728  0.935484  0.081723
5       0.75   0.049985  0.923311  0.094836
6       0.80   0.059357  0.908095  0.111431
7       0.85   0.073043  0.881315  0.134905
8       0.90   0.098270  0.850274  0.176178
9       0.95   0.187353  0.775411  0.301788

Best DL Threshold:
Threshold    0.950000
Precision    0.187353
Recall       0.775411
F1           0.301788
Name: 9, dtype: float64


In [25]:

from pathlib import Path

MODEL_PATH = Path("../models/dl_fraud_model.pkl")

torch.save(model.state_dict(), MODEL_PATH)

print("Model saved successfully!")
print(MODEL_PATH)

Model saved successfully!
..\models\dl_fraud_model.pkl


In [26]:
print(MODEL_PATH.exists())

True
